# Pipeline ETL End-to-End: Procesamiento de Vuelos (Arquitectura Medallón)
### Ingesta, Limpieza, Enriquecimiento y Agregaciones de Negocio

---

## 📌 Contexto del Proyecto
Este pipeline implementa una **Arquitectura Medallón** de tres capas para el procesamiento masivo de datos de tráfico aéreo estadounidense durante el año 2023:

1. 🥉 **Capa Bronce (Raw):**
   - Ingesta de datos crudos en formato CSV:
     - `us_flights_2023_*.csv` (12 meses de vuelos comerciales operados).
     - `us_cancelled_div_2023_*.csv` (12 meses de vuelos cancelados o desviados).
     - `airports_geolocation.csv` (metadatos geográficos y nombres oficiales de aeropuertos).
     - `weather_meteo_by_airport.csv` (registros climáticos diarios por aeropuerto: temperatura, lluvia, viento, nieve, presión).

2. 🥈 **Capa Plata (Cleansed / Conformed):**
   - Limpieza, desduplicación, normalización de tipos y esquemas.
   - Tratamiento riguroso de valores nulos e imputaciones basadas en dominio.
   - Unificación de vuelos operados y cancelados en una estructura columnar consistente.
   - Almacenamiento optimizado en formato **Apache Parquet** comprimido (`snappy`), reduciendo el consumo de memoria en más de un 80%.

3. 🥇 **Capa Oro (Aggregated / Business Features):**
   - Creación de datasets analíticos listos para dashboards ejecutivos (`Streamlit`) y matrices de características estructuradas para modelado predictivo de Machine Learning.


In [ ]:
import os
import sys
import glob
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')

# Configurar ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import config
config.ensure_dirs()

print("=" * 70)
print(f"Entorno configurado. Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio Bronce: {config.BRONZE_DIR}")
print(f"Directorio Plata:  {config.SILVER_DIR}")
print(f"Directorio Oro:    {config.GOLD_DIR}")
print("=" * 70)


## 1. Inspección de Archivos en Capa Bronce (Raw Data)
Verificamos la existencia de las fuentes de datos crudas antes de iniciar las transformaciones.


In [ ]:
# Listar fuentes disponibles
bronze_airports_geo = config.BRONZE_AIRPORTS / "airports_geolocation.csv"
bronze_weather = config.BRONZE_AIRPORTS / "weather_meteo_by_airport.csv"

flight_files = sorted(glob.glob(str(config.BRONZE_FLIGHTS / "*.csv")))
cancelled_files = sorted(glob.glob(str(config.BRONZE_CANCELLED / "*.csv")))

print(f"✓ Archivos de vuelos encontrados:      {len(flight_files)}")
print(f"✓ Archivos de cancelaciones/desvíos:   {len(cancelled_files)}")
print(f"✓ Archivo de geolocalización presente: {bronze_airports_geo.exists()}")
print(f"✓ Archivo meteorológico presente:      {bronze_weather.exists()}")


## 2. Procesamiento de Metadatos: Aeropuertos y Clima (Bronce -> Plata)
1. **Aeropuertos (`airports_geolocation.csv`)**:
   - Estandarización de códigos IATA en mayúsculas sin espacios.
   - Eliminación de registros duplicados.
   - Conversión a Parquet en `data/silver/airports.parquet`.

2. **Meteorología (`weather_meteo_by_airport.csv`)**:
   - Conversión de `time` a formato `datetime64[ns]`.
   - Normalización de identificadores IATA.
   - Imputación de variables de precipitación (`prcp`) y nieve (`snow`) con `0.0`.
   - Imputación de variables térmicas y de viento (`tavg`, `tmin`, `tmax`, `wspd`, `pres`) mediante mediana por aeropuerto.
   - Conversión a Parquet en `data/silver/weather.parquet`.


In [ ]:
# 2.1 Procesar Aeropuertos
print("Procesando aeropuertos (Bronce -> Plata)...")
df_airports = pd.read_csv(bronze_airports_geo)
df_airports['IATA_CODE'] = df_airports['IATA_CODE'].astype(str).str.strip().str.upper()
df_airports.drop_duplicates(subset=['IATA_CODE'], inplace=True)
df_airports.to_parquet(config.SILVER_AIRPORTS, index=False, engine='pyarrow')
print(f"-> Aeropuertos guardados en Plata: {len(df_airports)} aeropuertos únicos.")

# 2.2 Procesar Clima
print("
Procesando datos meteorológicos (Bronce -> Plata)...")
df_weather = pd.read_csv(bronze_weather)
df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather['airport_id'] = df_weather['airport_id'].astype(str).str.strip().str.upper()

# Imputación guiada por dominio
df_weather['prcp'] = df_weather['prcp'].fillna(0.0).astype('float32')
df_weather['snow'] = df_weather['snow'].fillna(0.0).astype('float32')

for col in ['tavg', 'tmin', 'tmax', 'wspd', 'pres']:
    if col in df_weather.columns:
        df_weather[col] = df_weather.groupby('airport_id')[col].transform(lambda x: x.fillna(x.median()))
        df_weather[col] = df_weather[col].fillna(df_weather[col].median()).astype('float32')

df_weather.to_parquet(config.SILVER_WEATHER, index=False, engine='pyarrow', compression='snappy')
print(f"-> Registros de clima guardados en Plata: {len(df_weather):,} observaciones.")


## 3. Procesamiento y Limpieza de Vuelos y Cancelaciones (Bronce -> Plata)
Para manejar de forma eficiente el volumen de datos de 2023 (~800 MB en CSV):
- Procesamos mes a mes para mantener bajo control el consumo de memoria RAM.
- Unificamos los vuelos operados con los vuelos cancelados/desviados agregando indicadores explícitos (`Cancelled`, `Diverted`).
- Corregimos tipos de datos a representaciones compactas (`int8`, `float32`, categorías).
- Cada mes se almacena de forma particionada en `data/silver/flights/flights_YYYY_MM.parquet`.


In [ ]:
def clean_and_unify_monthly_flights(flight_csv_path, cancelled_csv_path=None):
    """Limpia, tipa y fusiona los vuelos de un mes específico."""
    df_f = pd.read_csv(flight_csv_path, low_memory=False)
    df_f['Cancelled'] = 0
    df_f['Diverted'] = 0
    
    if cancelled_csv_path and Path(cancelled_csv_path).exists():
        df_c = pd.read_csv(cancelled_csv_path, low_memory=False)
        for col in ['Manufacturer', 'Model', 'Aircraft_age']:
            if col not in df_c.columns:
                df_c[col] = 'UNKNOWN' if col != 'Aircraft_age' else np.nan
        df_merged = pd.concat([df_f, df_c], ignore_index=True)
    else:
        df_merged = df_f

    # Conversión y normalización de fechas y enteros
    df_merged['FlightDate'] = pd.to_datetime(df_merged['FlightDate'])
    df_merged['Day_Of_Week'] = df_merged['Day_Of_Week'].astype('int8')
    df_merged['Cancelled'] = df_merged['Cancelled'].fillna(0).astype('int8')
    df_merged['Diverted'] = df_merged['Diverted'].fillna(0).astype('int8')

    # Retrasos y duraciones en minutos
    metric_cols = ['Dep_Delay', 'Arr_Delay', 'Flight_Duration',
                   'Delay_Carrier', 'Delay_Weather', 'Delay_NAS', 
                   'Delay_Security', 'Delay_LastAircraft']
    for col in metric_cols:
        if col in df_merged.columns:
            df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce').fillna(0.0).astype('float32')

    # Limpieza de cadenas de texto
    text_cols = ['airline', 'dep_airport', 'arr_airport', 'DepTime_label', 
                 'Dep_Delay_Type', 'Arr_Delay_Type', 'Distance_type']
    for col in text_cols:
        if col in df_merged.columns:
            df_merged[col] = df_merged[col].astype(str).str.strip().str.upper()

    return df_merged

# Ejecutar transformación mensual a Plata
total_rows_silver = 0
for f_path in flight_files:
    file_name = Path(f_path).name
    month_suffix = file_name.replace("us_flights_", "").replace(".csv", "")
    c_path = config.BRONZE_CANCELLED / f"us_cancelled_div_{month_suffix}.csv"
    
    df_month = clean_and_unify_monthly_flights(f_path, c_path if c_path.exists() else None)
    out_parquet = config.SILVER_FLIGHTS / f"flights_{month_suffix}.parquet"
    
    df_month.to_parquet(out_parquet, index=False, engine='pyarrow', compression='snappy')
    total_rows_silver += len(df_month)
    print(f"✓ Mes {month_suffix} procesado: {len(df_month):,} filas -> {out_parquet.name}")

print(f"
Total consolidado en Capa Plata: {total_rows_silver:,} registros.")


## 4. Construcción de la Capa Oro (Business Aggregations & Metrics)
A partir de los datos limpios en la Capa Plata, generamos tablas agregadas de alto rendimiento:

1. `daily_airport_metrics.parquet`:
   - Agregación por fecha y aeropuerto origen (vuelos totales, cancelados, desviados, retrasos medios, vuelos con retraso $\ge 15$ min).
   - Enriquecido con variables meteorológicas del día.
2. `airport_summary.parquet`:
   - KPIs consolidados a nivel de aeropuerto (volumen anual, tasa de cancelación, ranking y coordenadas GPS).
3. `airline_monthly_metrics.parquet`:
   - Desempeño mensual de puntualidad por aerolínea.
4. `route_metrics.parquet`:
   - Métricas de tráfico y retrasos por origen-destino.
5. `delay_causes_monthly.parquet`:
   - Distribución agregada de causas de retraso (Clima, Aerolínea, Control de Tráfico Aéreo NAS, Aeronave anterior, Seguridad).


In [ ]:
silver_parquet_files = sorted(glob.glob(str(config.SILVER_FLIGHTS / "*.parquet")))
print(f"Agregando métricas de {len(silver_parquet_files)} particiones de Plata...")

df_airports_silver = pd.read_parquet(config.SILVER_AIRPORTS)
df_weather_silver = pd.read_parquet(config.SILVER_WEATHER)

daily_list = []
airline_list = []
route_list = []
causes_list = []

for p_file in silver_parquet_files:
    chunk = pd.read_parquet(p_file)
    
    # 1. Diario por aeropuerto de salida
    d_agg = chunk.groupby(['FlightDate', 'dep_airport']).agg(
        total_flights=('FlightDate', 'count'),
        cancelled_flights=('Cancelled', 'sum'),
        diverted_flights=('Diverted', 'sum'),
        avg_dep_delay=('Dep_Delay', 'mean'),
        max_dep_delay=('Dep_Delay', 'max'),
        delayed_15m=('Dep_Delay', lambda x: (x >= 15).sum())
    ).reset_index()
    daily_list.append(d_agg)
    
    # 2. Mensual por aerolínea
    chunk['YearMonth'] = chunk['FlightDate'].dt.to_period('M').astype(str)
    a_agg = chunk.groupby(['YearMonth', 'airline']).agg(
        total_flights=('FlightDate', 'count'),
        cancelled_flights=('Cancelled', 'sum'),
        avg_dep_delay=('Dep_Delay', 'mean'),
        avg_arr_delay=('Arr_Delay', 'mean'),
        delayed_15m=('Dep_Delay', lambda x: (x >= 15).sum())
    ).reset_index()
    airline_list.append(a_agg)
    
    # 3. Métricas por ruta
    r_agg = chunk.groupby(['dep_airport', 'arr_airport']).agg(
        flight_count=('FlightDate', 'count'),
        avg_duration=('Flight_Duration', 'mean'),
        avg_dep_delay=('Dep_Delay', 'mean'),
        cancellation_rate=('Cancelled', 'mean')
    ).reset_index()
    route_list.append(r_agg)
    
    # 4. Causas de demoras
    c_agg = chunk[chunk['Dep_Delay'] > 0].groupby('YearMonth').agg(
        delay_carrier=('Delay_Carrier', 'sum'),
        delay_weather=('Delay_Weather', 'sum'),
        delay_nas=('Delay_NAS', 'sum'),
        delay_security=('Delay_Security', 'sum'),
        delay_last_aircraft=('Delay_LastAircraft', 'sum')
    ).reset_index()
    causes_list.append(c_agg)

# Consolidar y unir clima con la capa Oro
df_gold_daily = pd.concat(daily_list, ignore_index=True)
df_gold_daily = df_gold_daily.merge(
    df_weather_silver,
    left_on=['FlightDate', 'dep_airport'],
    right_on=['time', 'airport_id'],
    how='left'
).drop(columns=['time', 'airport_id'], errors='ignore')
df_gold_daily.to_parquet(config.GOLD_DAILY_AIRPORT, index=False)
print(f"✓ Oro Guardado: {config.GOLD_DAILY_AIRPORT.name}")

# Resumen de Aeropuertos
df_gold_airports = df_gold_daily.groupby('dep_airport').agg(
    total_departures=('total_flights', 'sum'),
    total_cancellations=('cancelled_flights', 'sum'),
    overall_avg_delay=('avg_dep_delay', 'mean'),
    severe_delayed_flights=('delayed_15m', 'sum')
).reset_index()
df_gold_airports['cancellation_rate'] = (df_gold_airports['total_cancellations'] / df_gold_airports['total_departures']) * 100
df_gold_airports = df_gold_airports.merge(df_airports_silver, left_on='dep_airport', right_on='IATA_CODE', how='inner')
df_gold_airports.to_parquet(config.GOLD_AIRPORT_SUMMARY, index=False)
print(f"✓ Oro Guardado: {config.GOLD_AIRPORT_SUMMARY.name}")

# Métricas por Aerolínea
df_gold_airline = pd.concat(airline_list, ignore_index=True).groupby(['YearMonth', 'airline']).sum().reset_index()
df_gold_airline['ontime_rate'] = (1 - (df_gold_airline['delayed_15m'] / df_gold_airline['total_flights'])) * 100
df_gold_airline['cancellation_rate'] = (df_gold_airline['cancelled_flights'] / df_gold_airline['total_flights']) * 100
df_gold_airline.to_parquet(config.GOLD_AIRLINE_MONTHLY, index=False)
print(f"✓ Oro Guardado: {config.GOLD_AIRLINE_MONTHLY.name}")

# Rutas
df_gold_routes = pd.concat(route_list, ignore_index=True).groupby(['dep_airport', 'arr_airport']).agg({
    'flight_count': 'sum',
    'avg_duration': 'mean',
    'avg_dep_delay': 'mean',
    'cancellation_rate': 'mean'
}).reset_index()
df_gold_routes.to_parquet(config.GOLD_ROUTES, index=False)
print(f"✓ Oro Guardado: {config.GOLD_ROUTES.name}")

# Causas de retrasos
df_gold_causes = pd.concat(causes_list, ignore_index=True).groupby('YearMonth').sum().reset_index()
df_gold_causes.to_parquet(config.GOLD_DELAY_CAUSES, index=False)
print(f"✓ Oro Guardado: {config.GOLD_DELAY_CAUSES.name}")


## 5. Preparación del Dataset de Features para Machine Learning (Gold Layer)
Creamos una muestra estratificada representativa que incluye variables meteorológicas, operativas y la variable objetivo binaria:
- `is_delayed_or_cancelled = 1` si el retraso en salida es $\ge 15$ minutos o el vuelo fue cancelado.
- `0` en caso contrario.


In [ ]:
print("Generando dataset de features para Machine Learning...")
ml_sample_list = []

for p_file in silver_parquet_files:
    chunk = pd.read_parquet(p_file, columns=[
        'FlightDate', 'Day_Of_Week', 'airline', 'dep_airport', 'arr_airport',
        'DepTime_label', 'Dep_Delay', 'Cancelled', 'Flight_Duration', 
        'Distance_type', 'Aircraft_age'
    ])
    chunk['is_delayed_or_cancelled'] = ((chunk['Dep_Delay'] >= 15) | (chunk['Cancelled'] == 1)).astype('int8')
    chunk['Month'] = chunk['FlightDate'].dt.month.astype('int8')
    chunk['Is_Weekend'] = chunk['Day_Of_Week'].isin([6, 7]).astype('int8')
    
    # Muestra de 25,000 registros por mes para un balance óptimo de representatividad y memoria
    sample_size = min(25000, len(chunk))
    ml_sample_list.append(chunk.sample(n=sample_size, random_state=42))

df_ml = pd.concat(ml_sample_list, ignore_index=True)

# Enriquecer con clima diario del aeropuerto de origen
df_ml = df_ml.merge(
    df_weather_silver[['time', 'airport_id', 'tavg', 'prcp', 'snow', 'wspd', 'pres']],
    left_on=['FlightDate', 'dep_airport'],
    right_on=['time', 'airport_id'],
    how='left'
).drop(columns=['time', 'airport_id'], errors='ignore')

# Imputación final de valores meteorológicos faltantes con la mediana
for col in ['tavg', 'prcp', 'snow', 'wspd', 'pres']:
    df_ml[col] = df_ml[col].fillna(df_ml[col].median()).astype('float32')

gold_ml_features_path = config.GOLD_DIR / "ml_features_dataset.parquet"
df_ml.to_parquet(gold_ml_features_path, index=False, engine='pyarrow', compression='snappy')
print(f"✓ Dataset de ML guardado: {len(df_ml):,} filas -> {gold_ml_features_path}")


## 6. Verificación y Resumen de la Arquitectura Medallón
Comprobamos el estado y tamaño de todos los artefactos persistidos en disco.


In [ ]:
print("=" * 80)
print("AUDITORÍA DE LA ARQUITECTURA MEDALLÓN")
print("=" * 80)
print(f"• Capa Bronce: {len(list(config.BRONZE_DIR.glob('**/*.csv')))} archivos CSV originales")
print(f"• Capa Plata:  {len(list(config.SILVER_DIR.glob('**/*.parquet')))} archivos Parquet limpios")
print(f"• Capa Oro:    {len(list(config.GOLD_DIR.glob('*.parquet')))} tablas analíticas consolidadas:")
for f in sorted(config.GOLD_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   -> {f.name:<35} | {size_mb:>6.2f} MB")
print("=" * 80)
print("Pipeline ETL completado satisfactoriamente.")
